In [1]:
from po_valley_methane_forecasting.paths import find_project_root

In [ ]:
from po_valley_methane_forecasting.rag.ingestion import chunk_corpus

from po_valley_methane_forecasting.rag.embedding import (
    load_embedding_model,
    embed_chunks
)

from po_valley_methane_forecasting.rag.reranking import(
    load_reranker,
    rerank_chunks
)

from po_valley_methane_forecasting.rag.retrieval import (
    retrieve_candidates,
    select_final_chunks
)

In [ ]:
from po_valley_methane_forecasting.rag.generation import (
    build_generation_prompt,
    generate_answer
)

In [ ]:
reranker= load_reranker()

In [5]:
project_root=find_project_root()

In [6]:
documents_dir = (
    project_root
    / "rag"
    / "documents"
)

all_chunks = chunk_corpus(
    documents_dir,
    max_chars=1000,
    overlap_sentences=1,
)

print("Documents:", len(
    {chunk["source"] for chunk in all_chunks}
))

print("Total chunks:", len(all_chunks))

Documents: 7
Total chunks: 852


In [7]:
from collections import Counter

source_counts = Counter(
    chunk["source"]
    for chunk in all_chunks
)

for source, count in source_counts.items():
    print(f"{source}: {count}")

global_assessment_of_oil_and_gas_methane_ultra_emitters.pdf: 33
global_tracking_quantification_of_oil_and_gas_methane_emissions_sentinel2_imagery.pdf: 60
high_resolution_assessment_of_coal_mining_methane_emissions_by_satellite_in_shanxi_china.pdf: 40
operational_methane_retrieval_algorithm_for_tropomi.pdf: 73
sentinel5_precursor_tropomi_level_2_product_user_manual_methane.pdf: 438
validation_methane_carbon_monoxide_from_sentinel5_precursor_using_TCCON_and_NDACC.pdf: 200
project_results.md: 8


In [ ]:
embedding_model = load_embedding_model()

corpus_embeddings = embed_chunks(
    chunks=all_chunks,
    model=embedding_model,
)

In [9]:
print(corpus_embeddings.shape)
print(corpus_embeddings.dtype)

(852, 384)
float32


In [ ]:
query_1 = (
    "Which forecasting model performed best "
    "on the final 2024 test?"
)

In [ ]:
candidates_chunks_1 = retrieve_candidates(
    query=query_1,
    model=embedding_model,
    corpus_embeddings=corpus_embeddings,
    chunks=all_chunks,
    candidate_k=50,
    min_score=0.35,
)

In [ ]:
reranked_chunks_1 = rerank_chunks(
    query=query_1,
    chunks=candidates_chunks_1,
    model=reranker,
)

In [ ]:
for chunk in reranked_chunks_1[:10]:
    print(
        f"Dense: {chunk['score']:.3f} | "
        f"Rerank: {chunk['rerank_score']:.3f} | "
        f"Source: {chunk['source']} | "
        f"Section: {chunk.get('section', '-')}"
    )

In [ ]:
results_1 = select_final_chunks(
    chunks=reranked_chunks_1,
    top_k=5,
    max_per_source=2,
)

In [ ]:
for result in results_1:
    print(result)

In [ ]:
prompt_1=build_generation_prompt(query_1, results_1)

In [ ]:
answer_1, metrics_1 = generate_answer(
    prompt_1
)

In [ ]:
print(answer_1)

The final 2024 test evaluation in the sources indicates that Ridge is the best-performing model, outperforming the persistence baseline on both MAE and RMSE. Ridge achieves a lower MAE by approximately 9.1% and RMSE by approximately 7.4% relative to persistence. This conclusion is directly supported by the data in SOURCE [SRC_1], which provides the specific values for MAE, RMSE, and R2 for both models.

The comparison between Ridge and persistence is based on the results from the final 2024 test period, which is part of the validation strategy described in SOURCE [SRC_2]. The validation strategy involves using two disjoint annual temporal validation folds (2022 and 2023) and training the best-performing model again on the 2019–2023 period to evaluate on the final 2024 test period. This approach ensures that the model is tested on a representative dataset, and the results from SOURCE [SRC_1] reflect this evaluation.

The performance metrics provided in SOURCE [SRC_1] are consistent with

In [ ]:
print(metrics_1)

{'prompt_tokens': 1541, 'output_tokens': 348, 'prompt_eval_duration': 62434798000, 'generation_duration': 89950690000, 'total_duration': 158622551700}


In [ ]:
tokens_per_second = (
    metrics_1["output_tokens"]
    / metrics_1["generation_duration"]
    * 1e9
)

print(f"{tokens_per_second:.2f} tokens/s")

In [ ]:
query_2 = (
    "What does Sentinel-5P XCH4 represent?"
)

In [ ]:
candidates_chunks_2 = retrieve_candidates(
    query=query_2,
    model=embedding_model,
    corpus_embeddings=corpus_embeddings,
    chunks=all_chunks,
    candidate_k=50,
    min_score=0.35,
)

In [ ]:
reranked_chunks_2 = rerank_chunks(
    query=query_2,
    chunks=candidates_chunks_2,
    model=reranker,
)

In [ ]:
for chunk in reranked_chunks_2[:10]:
    print(
        f"Dense: {chunk['score']:.3f} | "
        f"Rerank: {chunk['rerank_score']:.3f} | "
        f"Source: {chunk['source']} | "
        f"Section: {chunk.get('section', '-')}"
    )

In [ ]:
results_2 = select_final_chunks(
    chunks=reranked_chunks_2,
    top_k=5,
    max_per_source=2,
)

In [ ]:
for result in results_2:
    print(result)

In [ ]:
prompt_2=build_generation_prompt(query_2, results_2)

In [ ]:
answer_2, metrics_2 = generate_answer(
    prompt_2
)

In [ ]:
print(answer_2)

Sentinel-5P XCH4 represents the total column-averaged concentration of methane, which means it reflects the average methane concentration across the entire atmospheric column, not just at the surface. This product is derived from the TROPOMI instrument and is calculated using a column averaging kernel that accounts for the sensitivity of the retrieval to different altitude layers in the atmosphere. The column averaging kernel is essential for accurately representing the vertical distribution of methane, as it adjusts for the varying contributions from different atmospheric layers.

The XCH4 product is not directly measured as a surface concentration but is instead an inferred value based on retrievals from the satellite. The quality of the XCH4 product is assessed using the qa_value, which indicates the reliability of the retrieval output. To ensure the highest quality data, only pixels with qa_value > 0.5 are used. The precision of the XCH4 product is described as the standard deviati

In [ ]:
query_3 = (
    "Does the fact that the Ridge model outperforms the persistence baseline"
    "in the Po Valley XCH4 forecasting project imply that"
    "XCH4 evolves approximately linearly?"
)

In [ ]:
candidates_chunks_3 = retrieve_candidates(
    query=query_3,
    model=embedding_model,
    corpus_embeddings=corpus_embeddings,
    chunks=all_chunks,
    candidate_k=50,
    min_score=0.35,
)

In [ ]:
reranked_chunks_3 = rerank_chunks(
    query=query_3,
    chunks=candidates_chunks_3,
    model=reranker,
)

In [ ]:
for chunk in reranked_chunks_3[:10]:
    print(
        f"Dense: {chunk['score']:.3f} | "
        f"Rerank: {chunk['rerank_score']:.3f} | "
        f"Source: {chunk['source']} | "
        f"Section: {chunk.get('section', '-')}"
    )

In [ ]:
results_3 = select_final_chunks(
    chunks=reranked_chunks_3,
    top_k=5,
    max_per_source=2,
)

In [ ]:
for result in results_3:
    print(result)

In [ ]:
prompt_3=build_generation_prompt(query_3, results_3)

In [ ]:
answer_3, metrics_3 = generate_answer(
    prompt_3
)

In [ ]:
print(answer_3)

The results from the Po Valley XCH4 forecasting project indicate that the Ridge model outperforms the persistence baseline, but this does not necessarily imply that XCH4 evolves approximately linearly. The Ridge model's performance is attributed to its ability to regularize a linear combination of recent XCH4 history, spatial information, and seasonal features, which suggests a statistical relationship rather than a causal or intrinsic linear physical dependence. This conclusion is supported by the fact that the model's predictive relationship is not interpreted as evidence of a causal or linear physical process, as stated in Source [SRC_1].

The performance of the Ridge model in the Po Valley XCH4 forecasting project is also influenced by the uncertainty in XCH4 retrievals and atmospheric transport, as discussed in Source [SRC_3]. The relative standard deviation of posterior emissions due to errors in XCH4 retrievals is relatively small compared to the standard deviation due to the un

In [49]:
query_4 = (
    "What factors affect the accuracy of TROPOMI XCH4 retrievals?"
)

In [50]:
candidates_chunks_4 = retrieve_candidates(
    query=query_4,
    model=embedding_model,
    corpus_embeddings=corpus_embeddings,
    chunks=all_chunks,
    candidate_k=50,
    min_score=0.35,
)

In [51]:
reranked_chunks_4 = rerank_chunks(
    query=query_4,
    chunks=candidates_chunks_4,
    model=reranker,
)

In [52]:
results_4 = select_final_chunks(
    chunks=reranked_chunks_4,
    top_k=5,
    max_per_source=2,
)

In [53]:
prompt_4=build_generation_prompt(query_4, results_4)

In [54]:
answer_4, metrics_4 = generate_answer(
    prompt_4
)

In [55]:
print(answer_4)

The accuracy of TROPOMI XCH4 retrievals is influenced by several factors, including atmospheric input errors, retrieval parameters, and data processing techniques. Source [SRC_1] highlights that pressure errors can affect the retrieval process by altering the cross sections and the retrieved air column, leading to retrieval errors of similar magnitude. The perturbation of the prior pressure profile with a scaling factor up to ±0.3% is used to evaluate the net effect of pressure errors, indicating that pressure is a critical factor affecting accuracy. Source [SRC_2] further explains that the precision of XCH4 retrievals is determined by the standard deviation of retrieval noise, σXCH4, and that the signal-to-noise ratio becomes a limiting factor under specific conditions, such as snow-covered ground and large solar zenith angles (SZA). This suggests that atmospheric conditions and data quality play a significant role in the accuracy of XCH4 retrievals.

The precision of XCH4 retrievals 